In [73]:
import colorsys
from PIL import Image

width, height = 101, 101
img = Image.new("RGB", (width, height))

for y in range(height):
    for x in range(width):
        # Convert HSL (H=0, S=x/100, L=y/100) to RGB.
        # colorsys.hls_to_rgb expects (h, l, s).
        r, g, b = colorsys.hls_to_rgb(35/360, (100-y) / 100, x / 100)
        img.putpixel((x, y), (int(r * 255), int(g * 255), int(b * 255)))

img.save("orange.png")


In [7]:
import numpy as np

def compute_points(X, N):
    W = np.array([1, 1])
    G = np.array([0, 0.5])
    
    # Compute B (intersection with L=0)
    s_x, l_x = X
    slope = (l_x - W[1]) / (s_x - W[0])
    s_b = s_x - (l_x / slope)
    B = np.array([s_b, 0])
    
    # Compute N evenly spaced points between X and G
    points_XG = list(zip([t for t in np.linspace(G[0], s_x, N + 2)[1:-1]], [t for t in np.linspace(G[1], l_x, N + 2)[1:-1]]))
    
    # Compute N points between X and W in geometric progression
    ratios_XW = np.geomspace(1, np.linalg.norm(W - X), N + 1)[1:]
    points_XW = [tuple(X + (W - X) * (r / np.linalg.norm(W - X))) for r in ratios_XW]
    
    # Compute N points between X and B in geometric progression
    ratios_XB = np.geomspace(1, np.linalg.norm(B - X), N + 1)[1:]
    points_XB = [tuple(X + (B - X) * (r / np.linalg.norm(B - X))) for r in ratios_XB]
    
    return B, points_XG, points_XW, points_XB

# Example usage
X = (0.9, 0.6)
N = 3
B, points_XG, points_XW, points_XB = compute_points(X, N)

print("B:", B)
print("Points between X and G:", points_XG)
print("Points between X and W:", points_XW)
print("Points between X and B:", points_XB)


B: [0.75 0.  ]
Points between X and G: [(0.225, 0.525), (0.45, 0.55), (0.675, 0.575)]
Points between X and W: [(1.0805165505978112, 1.3220662023912448), (1.0343564477789626, 1.1374257911158507), (1.0, 1.0)]
Points between X and B: [(0.6933601335140863, -0.22655946594365484), (0.7239432478634032, -0.10422700854638678), (0.75, 0.0)]


In [4]:
(1,2)*3

(1, 2, 1, 2, 1, 2)

In [ ]:
def compute_points(sx, lx, N):
    W = (1.0, 1.0)
    G = (0.0, 0.5)
    X = (sx, lx)

    denom = (lx - 1.0)
    if abs(denom) < 1e-12:
        B = X
    else:
        t = -1.0 / denom
        sb = 1.0 + t*(sx - 1.0)
        B = (sb, 0.0)

    def interpolate_points(A, B, n):
        (ax, ay), (bx, by) = A, B
        return [
            (ax + (i/(n+1))*(bx - ax), ay + (i/(n+1))*(by - ay))
            for i in range(1, n+1)
        ]

    XG_points = interpolate_points(X, G, N)
    XW_points = interpolate_points(X, W, N)
    XB_points = interpolate_points(X, B, N)

    return {
        "B": B,
        "XG_points": XG_points,
        "XW_points": XW_points,
        "XB_points": XB_points
    }


B = (0.75, 0.0)
Points between X and G: [(0.75, 0.5833333333333333), (0.6000000000000001, 0.5666666666666667), (0.45, 0.55), (0.30000000000000004, 0.5333333333333333), (0.15000000000000002, 0.5166666666666666)]
Points between X and W: [(0.9166666666666667, 0.6666666666666666), (0.9333333333333333, 0.7333333333333333), (0.95, 0.8), (0.9666666666666667, 0.8666666666666667), (0.9833333333333334, 0.9333333333333333)]
Points between X and B: [(0.875, 0.5), (0.85, 0.4), (0.825, 0.3), (0.8, 0.2), (0.775, 0.09999999999999998)]


In [41]:

# Example for the figure with H=0, X=(s_x,l_x), N=3
result = compute_points(sx=0.9, lx=0.6, N=5)
print("B =", result["B"])
print("Points between X and G:", result["XG_points"])
print("Points between X and W:", result["XW_points"])
print("Points between X and B:", result["XB_points"])


B = (0.75, 0.0)
Points between X and G: [(0.75, 0.5833333333333333), (0.6000000000000001, 0.5666666666666667), (0.45, 0.55), (0.30000000000000004, 0.5333333333333333), (0.15000000000000002, 0.5166666666666666)]
Points between X and W: [(0.9860173612132674, 0.9440694448530697), (0.9710571514893336, 0.8842286059573344), (0.9550510257216822, 0.8202041028867287), (0.9379258605791103, 0.7517034423164414), (0.9196034204447799, 0.6784136817791195)]
Points between X and B: [(0.8790260418199012, 0.5161041672796045), (0.8565857272340004, 0.4263429089360017), (0.8325765385825232, 0.3303061543300931), (0.8068887908686655, 0.2275551634746621), (0.7794051306671699, 0.1176205226686794)]


In [35]:
from PIL import Image, ImageDraw

def draw_points(sx, lx, N, out_file="points.png"):
    points = compute_points(sx, lx, N)
    img_size = 101
    im = Image.new("RGB", (img_size, img_size), "white")
    d = ImageDraw.Draw(im)

    def place_dot(s, l):
        x = int(s*(img_size-1))
        y = int((1.0 - l)*(img_size-1))
        d.point((x, y), fill="black")

    place_dot(sx, lx)
    place_dot(1, 1)
    place_dot(0, 0.5)
    place_dot(points["B"][0], points["B"][1])
    for p in points["XG_points"]:
        place_dot(p[0], p[1])
    for p in points["XW_points"]:
        place_dot(p[0], p[1])
    for p in points["XB_points"]:
        place_dot(p[0], p[1])

    im.save(out_file)

draw_points(0.9, 0.7, 5, "points.png")


In [42]:
from PIL import Image
import colorsys
import numpy as np

def draw_hsl_matrix(matrix, file_name, square_size):
    """
    Draws a PNG file representing a color matrix.

    Parameters:
        matrix (list of list of tuples): A 2D array where each element is an (H, S, L) tuple.
        file_name (str): The name of the output PNG file.
        square_size (int): The size (NxN) of each square in pixels.
    """
    rows = len(matrix)
    cols = max([len(row) for row in matrix]) if rows > 0 else 0
    img_size = (cols * square_size, rows * square_size)

    img = Image.new("RGB", img_size)
    pixels = img.load()

    for r in range(rows):
        for c in range(len(matrix[r])):
            h, s, l = matrix[r][c]
            r_val, g_val, b_val = [int(x * 255) for x in colorsys.hls_to_rgb(h, l, s)]
            color = (r_val, g_val, b_val)

            for i in range(square_size):
                for j in range(square_size):
                    pixels[c * square_size + i, r * square_size + j] = color

    img.save(file_name, "PNG")


In [30]:
result

{'B': (0.75, 0.0),
 'XG_points': [(0.75, 0.5833333333333333),
  (0.6000000000000001, 0.5666666666666667),
  (0.45, 0.55),
  (0.30000000000000004, 0.5333333333333333),
  (0.15000000000000002, 0.5166666666666666)],
 'XW_points': [(0.9166666666666667, 0.6666666666666666),
  (0.9333333333333333, 0.7333333333333333),
  (0.95, 0.8),
  (0.9666666666666667, 0.8666666666666667),
  (0.9833333333333334, 0.9333333333333333)],
 'XB_points': [(0.875, 0.5),
  (0.85, 0.4),
  (0.825, 0.3),
  (0.8, 0.2),
  (0.775, 0.09999999999999998)]}

In [43]:
h = 0
draw_hsl_matrix([
    [(h,0.9,0.6)],
    [(h,s,l) for s,l in result['XG_points']],
    [(h,s,l) for s,l in result['XW_points']],
    [(h,s,l) for s,l in result['XB_points']]], 'test111.png', 20)

In [40]:
from PIL import Image, ImageDraw
import math

def compute_points(sx, lx, N):
    # Fixed reference points in HSL space
    W = (1.0, 1.0)
    G = (0.0, 0.5)
    X = (sx, lx)
    
    # Compute B: intersection of line through W and X with l=0.
    denom = lx - 1.0
    if abs(denom) < 1e-12:
        B = X
    else:
        t = -1.0 / denom
        sb = 1.0 + t * (sx - 1.0)
        B = (sb, 0.0)
    
    # Linear interpolation between X and G (evenly spaced)
    def linear_interpolate(A, B, n):
        return [
            (A[0] + (i/(n+1))*(B[0]-A[0]), A[1] + (i/(n+1))*(B[1]-A[1]))
            for i in range(1, n+1)
        ]
    XG_points = linear_interpolate(X, G, N)
    
    # Geometric interpolation for points along W->X and X->B.
    def distance(P, Q):
        return math.sqrt((P[0]-Q[0])**2 + (P[1]-Q[1])**2)
    
    L_left = distance(W, X)
    L_right = distance(X, B)
    
    # Determine common ratio r so that for N=1: r = sqrt(L_right / L_left),
    # and in general: r^(N+1) = L_right / L_left.
    if L_left < 1e-12:
        r_val = 1
    else:
        r_val = (L_right / L_left)**(1/(N+1))
    
    # Given endpoints A and B, return N points (excluding endpoints)
    # whose positions along A->B follow a geometric progression.
    def geometric_interpolate(A, B, n, r):
        points = []
        # Fraction for the i-th point: f = (1 - r^i) / (1 - r^(n+1))
        for i in range(1, n+1):
            if abs(r - 1) < 1e-12:
                f = i / (n+1)
            else:
                f = (1 - r**i) / (1 - r**(n+1))
            points.append((A[0] + f*(B[0]-A[0]), A[1] + f*(B[1]-A[1])))
        return points
    
    XW_points = geometric_interpolate(W, X, N, r_val)
    XB_points = geometric_interpolate(X, B, N, r_val)
    
    return {
        "B": B,
        "XG_points": XG_points,
        "XW_points": XW_points,
        "XB_points": XB_points
    }


# Example with X=(0.4, 0.7) and N=3
draw_points(0.95, 0.8, 5, "points.png")


In [111]:
import math

def compute_points(sx, lx, N):
    W = (1.0, 1.0)
    B1 = (1.0, 0.0)
    G = (0.0, 0.5)
    X = (sx, lx)

    # Compute B: intersection of line through W and X with l=0.
    denom = lx - 1.0
    if abs(denom) < 1e-12:
        B = X
    else:
        t = -1.0 / denom
        sb = 1.0 + t * (sx - 1.0)
        B = (max(sb,0.0), 0.0)
        
    if abs(lx) < 1e-12:
        W1 = X
    else:
        t2 = 1.0 / lx
        s_w1 = 1.0 + t2 * (sx - 1.0)
        W1 = (max(s_w1,0.0), 1.0)

    # Euclidean distance
    def dist(A, B):
        return math.sqrt((A[0] - B[0])**2 + (A[1] - B[1])**2)

    # Geometric interpolation (excludes endpoints)
    def geometric_interpolate(A, B, n, r):
        pts = []
        for i in range(1, n+1):
            if abs(r - 1) < 1e-12:
                f = i / (n+1)
            else:
                f = (1 - r**i) / (1 - r**(n+1))
            pts.append((A[0] + f*(B[0]-A[0]), A[1] + f*(B[1]-A[1])))
        return pts

    # Compute ratio for geometric interpolation
    L_left = dist((X[0],1), X)
    L_right = dist(X, (X[0],0))
    if L_left < 1e-12:
        r_val = 1.0
    else:
        r_val = (L_right / L_left)**(1/(N+1))

    # Collect points along W->X->B
    WX_pts = geometric_interpolate(W1, X, N, r_val)
    XB_pts = geometric_interpolate(X, B, N, r_val)
    WB_line = WX_pts + [X] + XB_pts

    # Linear interpolation including endpoints
    def linear_including_endpoints(A, B, n):
        return [
            (
                A[0] + (i/(n+1))*(B[0]-A[0]),
                A[1] + (i/(n+1))*(B[1]-A[1])
            )
            for i in range(n+2)
        ]

    # For each point on the W->B line, create a row from G to that point
    green_lines = [linear_including_endpoints(G,(0,1),N)]+\
        [linear_including_endpoints(G, p, N) for p in WB_line]+\
        [linear_including_endpoints(G,(0,0),N)]

    return green_lines


In [121]:
result = compute_points(0.51, 0.29, 4)
result

[[(0.0, 0.5), (0.0, 0.6), (0.0, 0.7), (0.0, 0.8), (0.0, 0.9), (0.0, 1.0)],
 [(0.0, 0.5),
  (0.028271098708358828, 0.5606421959158142),
  (0.056542197416717656, 0.6212843918316284),
  (0.08481329612507647, 0.6819265877474425),
  (0.11308439483343531, 0.7425687836632567),
  (0.14135549354179414, 0.803210979579071)],
 [(0.0, 0.5),
  (0.051906915597956786, 0.527737431226374),
  (0.10381383119591357, 0.5554748624527478),
  (0.15572074679387035, 0.5832122936791218),
  (0.20762766239182714, 0.6109497249054956),
  (0.2595345779897839, 0.6386871561318695)],
 [(0.0, 0.5),
  (0.07166744368360307, 0.5002276764404742),
  (0.14333488736720615, 0.5004553528809483),
  (0.2150023310508092, 0.5006830293214224),
  (0.2866697747344123, 0.5009107057618966),
  (0.35833721841801536, 0.5011383822023707)],
 [(0.0, 0.5),
  (0.08818806880299401, 0.477228374803675),
  (0.17637613760598803, 0.45445674960735005),
  (0.26456420640898204, 0.431685124411025),
  (0.35275227521197605, 0.40891349921470005),
  (0.44094034

In [122]:
h = 131/360
draw_hsl_matrix([[(h,s,l) for s,l in row] for row in result], 'test222.png', 20)

In [108]:
def draw_point_matrix(matrix, out_file="points.png"):
    img_size = 101
    im = Image.new("RGB", (img_size, img_size), "red")
    d = ImageDraw.Draw(im)

    def place_dot(s, l):
        x = int(s*(img_size-1))
        y = int((1.0 - l)*(img_size-1))
        d.point((x, y), fill="black")

    for row in matrix:
        for x,y in row:
            place_dot(x,y)

    im.save(out_file)

In [128]:
draw_point_matrix(result)

In [126]:
draw_hsl_matrix([[(h,-s,1-l) for s,l in row] for row in reversed(result)], 'test333.png', 20)